In [ ]:
#!/usr/bin/env python3
"""
Plot MM/GBSA running-average results from an NPZ file.

Expected NPZ keys for each system:
    sys_0_time
    sys_0_mean
    sys_0_std
    sys_0_label
    sys_0_color

    sys_1_time
    sys_1_mean
    ...

Example:
    python plot_mmgbsa_running_average.py \
        --input data/processed/mmgbsa/mmgbsa_analysis_data.npz \
        --output results/figures/mmgbsa/mmgbsa_na_running_average.png \
        --ion Na \
        --analysis-start 50 \
        --xlim 0 100 \
        --ylim -25 1
"""

import argparse
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# 舊版資料中可能使用的簡寫與實際 docking score 對照
DOCKING_SCORE_MAP = {
    "310": "3.10",
    "360": "3.60",
    "412": "4.12",
    "436": "4.36",
    "564": "5.64",
    "606": "6.06",
    "687": "6.87",
    "688": "6.88",
    "688_2": "6.88_2",
    "721": "7.21",
}


def parse_arguments():
    """讀取命令列參數。"""

    parser = argparse.ArgumentParser(
        description=(
            "Plot MM/GBSA running-average free energies and "
            "standard-deviation regions."
        )
    )

    parser.add_argument(
        "--input",
        required=True,
        type=Path,
        help="Input NPZ file generated by the MM/GBSA analysis script.",
    )

    parser.add_argument(
        "--output",
        required=True,
        type=Path,
        help="Output figure path, such as mmgbsa_running_average.png.",
    )

    parser.add_argument(
        "--ion",
        choices=["K", "Na"],
        default="Na",
        help="Ion environment displayed in the legend. Default: Na.",
    )

    parser.add_argument(
        "--analysis-start",
        type=float,
        default=50.0,
        help="Beginning of the shaded analysis region in ns. Default: 50.",
    )

    parser.add_argument(
        "--xlim",
        nargs=2,
        type=float,
        metavar=("XMIN", "XMAX"),
        default=(0.0, 100.0),
        help="X-axis range in ns. Default: 0 100.",
    )

    parser.add_argument(
        "--ylim",
        nargs=2,
        type=float,
        metavar=("YMIN", "YMAX"),
        default=(-25.0, 1.0),
        help="Y-axis range in kcal/mol. Default: -25 1.",
    )

    parser.add_argument(
        "--dpi",
        type=int,
        default=300,
        help="Resolution of the saved figure. Default: 300.",
    )

    parser.add_argument(
        "--legend-columns",
        type=int,
        default=2,
        help="Number of legend columns. Default: 2.",
    )

    parser.add_argument(
        "--no-show",
        action="store_true",
        help="Save the figure without displaying it.",
    )

    return parser.parse_args()


def convert_np_value(value):
    """
    將 NumPy scalar、零維陣列或 bytes 轉換為一般 Python 字串。
    """

    array = np.asarray(value)

    if array.ndim == 0:
        value = array.item()

    if isinstance(value, bytes):
        value = value.decode("utf-8")

    return str(value).strip()


def natural_prefix_sort(prefix):
    """依照 sys_0、sys_1、sys_2 的數字順序排列。"""

    match = re.search(r"(\d+)$", prefix)

    if match:
        return int(match.group(1))

    return float("inf")


def find_system_prefixes(data):
    """
    從 NPZ keys 自動找出所有系統前綴。

    例如：
        sys_0_label → sys_0
        sys_1_label → sys_1
    """

    prefixes = []

    for key in data.files:
        match = re.fullmatch(r"(sys_\d+)_label", key)

        if match:
            prefixes.append(match.group(1))

    return sorted(set(prefixes), key=natural_prefix_sort)


def clean_docking_label(raw_label):
    """
    統一 docking score 標籤格式，並相容舊版簡寫。

    Examples:
        606                 → Docking score=6.06
        K_606               → Docking score=6.06
        Docking score=6.06  → Docking score=6.06
    """

    label = raw_label.strip()

    # 如果原本已經是完整標籤，直接保留
    if label.lower().startswith("docking score="):
        return label

    # 優先處理 688_2，避免先匹配成 688
    ordered_codes = sorted(
        DOCKING_SCORE_MAP,
        key=len,
        reverse=True,
    )

    for code in ordered_codes:
        if code in label:
            score = DOCKING_SCORE_MAP[code]
            return f"Docking score={score}"

    # 如果輸入本身就是小數 docking score
    if re.fullmatch(r"\d+\.\d+", label):
        return f"Docking score={label}"

    # 無法辨認時保留原始文字，避免程式中斷
    return label


def validate_system_data(data, prefix):
    """檢查每個系統所需的資料是否完整。"""

    required_keys = [
        f"{prefix}_time",
        f"{prefix}_mean",
        f"{prefix}_std",
        f"{prefix}_label",
    ]

    missing_keys = [
        key for key in required_keys
        if key not in data.files
    ]

    if missing_keys:
        print(
            f"[警告] 跳過 {prefix}，缺少欄位："
            f"{', '.join(missing_keys)}"
        )
        return False

    time_axis = np.asarray(data[f"{prefix}_time"])
    mean_values = np.asarray(data[f"{prefix}_mean"])
    std_values = np.asarray(data[f"{prefix}_std"])

    if not (
        len(time_axis)
        == len(mean_values)
        == len(std_values)
    ):
        print(
            f"[警告] 跳過 {prefix}：time、mean、std 長度不一致。"
        )
        return False

    if len(time_axis) == 0:
        print(f"[警告] 跳過 {prefix}：資料為空。")
        return False

    return True


def plot_system(ax, data, prefix, ion):
    """繪製一個系統的平均值與標準差範圍。"""

    if not validate_system_data(data, prefix):
        return False

    time_axis = np.asarray(
        data[f"{prefix}_time"],
        dtype=float,
    )
    mean_values = np.asarray(
        data[f"{prefix}_mean"],
        dtype=float,
    )
    std_values = np.asarray(
        data[f"{prefix}_std"],
        dtype=float,
    )

    raw_label = convert_np_value(
        data[f"{prefix}_label"]
    )
    docking_label = clean_docking_label(raw_label)

    # color 為選用欄位；沒有指定時交由 Matplotlib 自動配色
    color_key = f"{prefix}_color"

    if color_key in data.files:
        color = convert_np_value(data[color_key])
    else:
        color = None

    # 使用 Unicode 上標，避免 LaTeX 與底線造成格式錯誤
    legend_label = f"{ion}⁺ ({docking_label})"

    lower_values = mean_values - std_values
    upper_values = mean_values + std_values

    line = ax.plot(
        time_axis,
        mean_values,
        linestyle="-",
        linewidth=3,
        alpha=0.9,
        color=color,
        label=legend_label,
    )[0]

    # 如果沒有預先指定顏色，使用平均線自動取得的顏色
    fill_color = line.get_color()

    ax.fill_between(
        time_axis,
        lower_values,
        upper_values,
        color=fill_color,
        alpha=0.15,
        linewidth=0,
    )

    print(
        f"繪製完成：{prefix}, "
        f"label={legend_label}, "
        f"frames={len(time_axis)}"
    )

    return True


def configure_axes(ax, args):
    """設定座標軸、分析區域、格線與字體。"""

    ax.set_title(
        "MM/GBSA Free Energy Comparison",
        fontsize=32,
        fontweight="bold",
        pad=20,
    )

    ax.set_xlabel(
        "Time (ns)",
        fontsize=32,
        fontweight="bold",
        labelpad=15,
    )

    ax.set_ylabel(
        "Free Energy (kcal/mol)",
        fontsize=32,
        fontweight="bold",
        labelpad=15,
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=26,
    )

    ax.set_xlim(args.xlim)
    ax.set_ylim(args.ylim)

    ax.grid(
        True,
        linestyle="--",
        alpha=0.6,
    )

    # 僅在 analysis-start 位於繪圖範圍內時加入陰影
    if args.xlim[0] < args.analysis_start < args.xlim[1]:
        ax.axvspan(
            args.analysis_start,
            args.xlim[1],
            color="grey",
            alpha=0.1,
            label="_nolegend_",
        )

        ax.axvline(
            args.analysis_start,
            color="grey",
            linestyle=":",
            linewidth=2,
            alpha=0.8,
        )

    ax.legend(
        fontsize=18,
        loc="lower left",
        frameon=True,
        framealpha=0.9,
        ncol=args.legend_columns,
    )


def main():
    """主程式。"""

    args = parse_arguments()

    if not args.input.exists():
        raise FileNotFoundError(
            f"找不到輸入檔案：{args.input}"
        )

    print(f"載入資料：{args.input}")
    print(f"離子環境：{args.ion}⁺")

    # 預設不允許載入 pickle，提高資料檔安全性
    with np.load(args.input, allow_pickle=False) as data:
        system_prefixes = find_system_prefixes(data)

        if not system_prefixes:
            raise ValueError(
                "NPZ 中找不到 sys_0_label、sys_1_label 等系統欄位。"
            )

        print(
            "找到系統："
            + ", ".join(system_prefixes)
        )

        fig, ax = plt.subplots(
            figsize=(13, 10),
            dpi=150,
        )

        plot_count = 0

        for prefix in system_prefixes:
            success = plot_system(
                ax=ax,
                data=data,
                prefix=prefix,
                ion=args.ion,
            )

            if success:
                plot_count += 1

    if plot_count == 0:
        plt.close(fig)
        raise ValueError("沒有有效資料可以繪製。")

    configure_axes(ax, args)

    plt.tight_layout()

    # 自動建立輸出資料夾
    args.output.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.savefig(
        args.output,
        dpi=args.dpi,
        bbox_inches="tight",
    )

    print(f"圖片已儲存：{args.output}")

    if args.no_show:
        plt.close(fig)
    else:
        plt.show()


if __name__ == "__main__":
    main()